# Iterators in Python — one item at a time

> **Explain it like I am five:** Imagine a Pez dispenser. You ask for **one candy**, it gives the next one, and it remembers where it stopped. An iterator behaves the same way.

This notebook keeps the original course examples and adds clearer explanations, visual mental models, practical patterns, common mistakes, practice, and a revision cheat sheet.

## Learning goals

By the end, you will be able to:

- tell the difference between an **iterable** and an **iterator**;
- use `iter()` and `next()`;
- explain `StopIteration` and how a `for` loop handles it;
- understand why iterators save memory;
- build a custom iterator with `__iter__()` and `__next__()`.


## 1. Start with a normal `for` loop

A list is like a toy box: it holds all its items. A `for` loop asks the box for an iterator and repeatedly requests the next item.

**Original example:**


In [1]:
my_list=[1,2,3,4,5,6]
for i in my_list:
    print(i)


1
2
3
4
5
6


`my_list` is a `list`. It is **iterable**, which means Python knows how to start walking through it. The list itself is not the walking process.


In [2]:
type(my_list)


list

In [3]:
print(my_list)


[1, 2, 3, 4, 5, 6]


## 2. Iterable vs. iterator

| Word | Tiny meaning | Examples |
|---|---|---|
| **Iterable** | Something we can start looping over | list, tuple, string, set, dictionary, file |
| **Iterator** | The object that remembers the current position | result of `iter(my_list)` |

Think of a **book** as an iterable and a **bookmark** as an iterator. The book contains the pages; the bookmark remembers which page comes next.

Calling `iter(iterable)` asks for an iterator.


In [4]:
## Iterator
iterator=iter(my_list)
print(type(iterator))


<class 'list_iterator'>


In [5]:
iterator


The strange-looking value such as `<list_iterator at ...>` is normal. It is the iterator object, not the list values themselves.

Calling `next(iterator)` asks for exactly one value and moves the invisible bookmark forward.


In [6]:
## Iterate through all the element

next(iterator)


1

> **Notebook tip:** An iterator has memory. If you run a `next()` cell several times, you get a new item each time. Re-run the next cell to create a fresh iterator from the beginning.


In [7]:
iterator=iter(my_list)


In [8]:
try:
    print(next(iterator))
except StopIteration:
    print("There are no elements in the iterator")


1


## 3. What a `for` loop secretly does

Python roughly turns this:

```python
for item in my_list:
    print(item)
```

into the following `while` loop. When there are no more items, `next()` raises `StopIteration`. The `for` loop catches it for us and stops quietly.


In [9]:
demo_iterator = iter(my_list)

while True:
    try:
        item = next(demo_iterator)
        print(item)
    except StopIteration:
        print("Finished: the iterator has no more items.")
        break


1
2
3
4
5
6
Finished: the iterator has no more items.


## 4. Strings are iterable too

A string gives one character at a time. This is the original string example.


In [10]:
# String iterator
my_string = "Hello"
string_iterator = iter(my_string)

print(next(string_iterator))  # Output: H
print(next(string_iterator))  # Output: e


H
e


## 5. Iterators are one-way and get exhausted

Most iterators are like a one-way ticket. After an item is used, the iterator does not rewind. Create a new iterator to start again.


In [11]:
numbers_iterator = iter([10, 20, 30])

print(list(numbers_iterator))  # Uses every remaining item.
print(list(numbers_iterator))  # Empty: the same iterator is exhausted.
print(list(iter([10, 20, 30])))  # Fresh iterator, so values appear again.


[10, 20, 30]
[]
[10, 20, 30]


## 6. Why use iterators?

A list normally stores all its values at once. An iterator can produce or reveal one item at a time.

- **Less memory:** useful for large files or large data streams.
- **Start quickly:** we do not always need to build the entire result first.
- **Composable:** iterators work nicely with `map`, `filter`, `zip`, and generators.

`map()` and `filter()` return lazy iterator-like objects in Python 3. Nothing is printed until the loop asks for values.


In [12]:
doubled = map(lambda number: number * 2, [1, 2, 3, 4])
even_only = filter(lambda number: number % 2 == 0, doubled)

print(even_only)       # The lazy object itself
print(list(even_only)) # Ask for all remaining results


[2, 4, 6, 8]


## 7. Helpful iterator tools

These tools avoid manual index bookkeeping:

- `enumerate(values, start=...)` gives `(index, value)` pairs;
- `zip(a, b)` pairs matching positions and stops at the shorter input;
- `reversed(values)` walks backwards without changing the original;
- `iter(callable, sentinel)` repeatedly calls a function until a stop value appears.


In [13]:
names = ["Asha", "Ben", "Chen"]
scores = [92, 85, 88]

print("enumerate:", list(enumerate(names, start=1)))
print("zip:", list(zip(names, scores)))
print("reversed:", list(reversed(names)))


enumerate: [(1, 'Asha'), (2, 'Ben'), (3, 'Chen')]
zip: [('Asha', 92), ('Ben', 85), ('Chen', 88)]
reversed: ['Chen', 'Ben', 'Asha']


In [14]:
# iter(callable, sentinel): stop when the callable returns the sentinel.
answers = iter(["yes", "maybe", "stop", "ignored"])
until_stop = iter(lambda: next(answers), "stop")
print(list(until_stop))


['yes', 'maybe']


## 8. Build a custom iterator

An iterator follows the **iterator protocol**:

1. `__iter__()` returns the iterator object.
2. `__next__()` returns the next value.
3. When finished, `__next__()` raises `StopIteration`.

The double-underscore methods are called **dunder methods**. Usually we call `iter(obj)` and `next(obj)` instead of calling the dunder methods directly.


In [15]:
class Countdown:
    """An iterator that counts down to 1."""

    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration

        value = self.current
        self.current -= 1
        return value


countdown = Countdown(3)
print(next(countdown))
print(next(countdown))
print(next(countdown))

try:
    print(next(countdown))
except StopIteration:
    print("Blast off!")


3
2
1
Blast off!


A `for` loop makes the same object easier to use:


In [16]:
for number in Countdown(5):
    print(number)


5
4
3
2
1


## 9. Infinite iterators — take only what you need

Some iterators never finish. `itertools.count()` keeps counting forever, so we combine it with `islice()` to take a safe number of values.


In [17]:
from itertools import count, islice

counting_forever = count(start=10, step=5)
first_five = islice(counting_forever, 5)
print(list(first_five))


[10, 15, 20, 25, 30]


## 10. Common mistakes

### Mistake 1: expecting an exhausted iterator to restart

Create a new iterator with `iter(original_iterable)`.

### Mistake 2: calling `next()` without handling the end

Use a `for` loop when possible, or catch `StopIteration` when manually calling `next()`.

### Mistake 3: expecting an iterator to support indexing

`iterator[0]` usually fails because an iterator only promises “give me the next item.” Convert to a list only if the data is small enough and you truly need indexing.

### Mistake 4: inspecting an iterator consumes it

`list(iterator)` is not just a peek—it uses every remaining item.


## 11. Mini practice

Try to predict each answer before running the cell.


In [18]:
# Practice 1: manually consume two colors.
colors = ["red", "green", "blue"]
color_iterator = iter(colors)

print(next(color_iterator))
print(next(color_iterator))


red
green


In [19]:
# Practice 2: pair each task with a human-friendly number.
tasks = ["Learn iterables", "Use next", "Handle the end"]

for position, task in enumerate(tasks, start=1):
    print(f"{position}. {task}")


1. Learn iterables
2. Use next
3. Handle the end


## Easy revision cheat sheet

| Need | Write | Remember |
|---|---|---|
| Get an iterator | `it = iter(values)` | Creates the bookmark |
| Get one item | `next(it)` | Moves the bookmark |
| Loop safely | `for x in values:` | Handles `StopIteration` automatically |
| Restart | `it = iter(values)` | Most iterators do not rewind |
| Build an iterator | `__iter__` + `__next__` | Raise `StopIteration` at the end |
| Get positions | `enumerate(values, 1)` | Cleaner than a manual counter |
| Pair inputs | `zip(a, b)` | Stops at the shortest input |

**One-sentence memory trick:** An **iterable** can give you a bookmark; an **iterator** is the bookmark that gives one item at a time.

**Interview check:** `iter(iterator) is iterator` is normally `True`, while `iter(list) is list` is `False` because a list creates a separate iterator.


In [20]:
sample_list = [1, 2]
sample_iterator = iter(sample_list)

print(iter(sample_iterator) is sample_iterator)
print(iter(sample_list) is sample_list)


True
False
